In [4]:
from openai import OpenAI
openai_client = OpenAI()

from dotenv import load_dotenv
load_dotenv()

True

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
question = "What is the exact percentage score required on CS50 problem sets to get a certificate?"
answer = llm(question)
print(answer)

CS50 does **not** award a certificate based on a required percentage score on problem sets.

Instead, for the free CS50 certificate, you generally need to:

- **Pass all problem sets** by meeting their specified **minimum requirements**
- Complete the **final project**
- Earn a **passing overall result** according to the course’s requirements

So there isn’t one exact “percentage on problem sets” threshold like “you need 70%.” The grading is typically **requirements-based**, not a single percentage cutoff.

If you mean a specific CS50 version, like **CS50x**, **CS50P**, or a verified/edX certificate, tell me which one and I can give the exact requirement for that course.


In [14]:
# TODO: fill in the rest of detials here so I can run the below to show in HOOK.

from minsearch import Index
import json

# 1. Load the local JSON file directly
with open("cs50_faq.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

# 2. Check how many documents were loaded
len(documents)

index = Index(
    text_fields=["question", "answer"],
    keyword_fields=["category"]
)

index.fit(documents)

index.search(question)  
index.search(question, filter_dict={'category':'Certificates'})  # returns all the questions only from the Certificates section


# The instructions tell the LLM its role and how to answer:

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""



def search(question, category='Certificates'):
    boost_dict={'question':2.0}
    filter_dict={'category':category}
    
    return index.search(question,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

search_results = search(question)  # note that this search() function is the minsearch's Index() function and not
                                   # the search() function we defined just above.

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["category"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text


prompt = build_prompt(question, search_results)
question = 'What is the exact percentage score required on CS50 problem sets to get a certificate?' 


In [10]:
# TO SHOW at the start - FULL RAG
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


In [15]:
answer = rag("What is the exact percentage score required on the CS50 problem sets to get a certificate?")
answer

'You must score **at least 70%** on each CS50 problem set to get a verified certificate.'